# Evaluation of the hierarchy attribute extraction pipeline

Here we evaluate the performance of **Step 2** of the Ariadne pipeline: extracting SNOMED CT hierarchy attributes for medical terms.

The pipeline:
1. Retrieves similar reference SNOMED terms (few-shot examples)
2. Uses an LLM to extract attribute components from the medical term
3. Retrieves candidate concepts for each extracted component via vector search
4. Uses an LLM to select the best candidate for each attribute

## Setup

Before running this notebook, make sure the environment is set up as described in the README.
This includes:
- Credentials for a database with the OHDSI Vocabulary loaded (including `snomed_attribute` and `snomed_reference` tables)
  If you need to recreate these two tables, please run /sandbox/build_pg_indexes_attributes.py
- Credentials for the LLM API
- A `config.yaml` with the hierarchy prompts configured

All results will be stored in the `data/notebook_results` folder.
Most code blocks will load results from file if they already exist, to save time and costs.
Delete those files to rerun the corresponding steps.

## Configuration

Load the hierarchy pipeline configuration from `config.yaml`.

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

project_root = Path.cwd().parent.parent
load_dotenv()

from ariadne.hierarchy.config import HierarchyConfig

cfg = HierarchyConfig.from_yaml(project_root / "config.yaml")
print(f"Extraction model: {cfg.models.extraction}")
print(f"Selection model: {cfg.models.selection}")
print(f"Reference examples: {cfg.retrieval.num_reference_examples}")
print(f"Top-k per category: {cfg.retrieval.top_k_per_category}")

Extraction model: o3
Selection model: o3
Reference examples: 5
Top-k per category: 20


## Gold standard

We use the hierarchy attributes gold standard file.
Each row maps a source concept to an expected SNOMED attribute (e.g. finding site, associated morphology).

This gold standard contains the following columns:
- `concept_id_1`: the concept ID of the source term
- `concept_name_1`: the source term text
- `concept_id_2`: the concept ID of the expected attribute value (e.g. a finding site concept)
- `concept_name_2`: the name of the expected attribute value
- `attribute_category`: the SNOMED relationship type (e.g. `Has finding site`, `Has asso morph`, `Has causative agent`)

In [2]:
gold_standard_path = project_root / "data" / "gold_standards" / "hierarchy_attributes_train_test_gs.csv"
# Alternative 1: held-out validation set created by selecting a random set of SNOMED terms
# script to create the csv is at sandbox/build_validation_gs.py
# gold_standard_path = project_root / "data" / "gold_standards" / "hierarchy_attributes_validation_gs.csv"
# Alternative 2: use unmatched terms from the exact matching pipeline (Step 1)
# gold_standard_path = project_root / "data" / "notebook_results" / "exact_matching_vector_search_results.csv"

gold_standard = pd.read_csv(gold_standard_path)

# If using exact_matching_vector_search_results.csv, filter to unmatched terms only:
# gold_standard = gold_standard[gold_standard["mapped_concept_id"] == -1]

print(f"Gold standard: {len(gold_standard)} rows")
unique_terms = gold_standard[["concept_id_1", "concept_name_1"]].drop_duplicates()
print(f"Unique terms: {len(unique_terms)}")
gold_standard.head(10)

Gold standard: 935 rows
Unique terms: 342


,concept_id_1,concept_name_1,concept_id_2,concept_name_2,attribute_category
0,193782,End-stage renal disease,4125554,Impaired,Has interpretation
1,193782,End-stage renal disease,4236661,Chronic,Has clinical course
2,193782,End-stage renal disease,4271678,Kidney structure,Has finding site
3,193782,End-stage renal disease,40282772,Renal function,Has interprets
4,375736,"Trachoma, active stage",4053015,Inflammation,Has asso morph
5,375736,"Trachoma, active stage",4105319,Conjunctival structure,Has finding site
6,375736,"Trachoma, active stage",4253008,Inflammatory morphology,Has asso morph
7,375736,"Trachoma, active stage",4274488,Chlamydia trachomatis,Has causative agent
8,375736,"Trachoma, active stage",40480911,Infectious process,Has pathology
9,443611,Chronic kidney disease stage 5,4125554,Impaired,Has interpretation


## Run the hierarchy attribute extraction pipeline

We use `process_gold_standard` from the evaluator module, which:
1. Iterates over each unique term in the gold standard
2. Runs the four-step attribute extraction pipeline (`find_attributes_two_stage`)
3. Supports checkpointing (resumes from where it left off if interrupted)


In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s")

from ariadne.hierarchy.evaluator import process_gold_standard
from ariadne.hierarchy.evaluator import _build_prediction_rows
from ariadne.hierarchy.searchers import SnomedAttributeSearcher, SnomedReferenceSearcher

raw_results_file = project_root / "data" / "notebook_results" / "hierarchy_results_raw.json"
os.makedirs(project_root / "data" / "notebook_results", exist_ok=True)

if raw_results_file.exists():
    with open(raw_results_file, "r") as f:
        results = json.load(f)
    print(f"Loaded {len(results)} cached results from file.")
else:
    # Exclude gold standard terms from reference examples to prevent data leakage
    gs_concept_ids = set(gold_standard["concept_id_1"].unique())
    with SnomedAttributeSearcher(cfg=cfg) as attr_idx, \
         SnomedReferenceSearcher(cfg=cfg, exclude_concept_ids=gs_concept_ids) as ref_idx:
        results = process_gold_standard(
            str(gold_standard_path),
            attr_idx,
            reference_index=ref_idx,
            cfg=cfg,
            max_workers=8,  
        )
    # Save raw results for debugging
    with open(raw_results_file, "w") as f:
        json.dump(results, f, indent=2, default=str)
    print(f"Processed {len(results)} terms. Results saved.")

total_cost = sum(r.get("cost", {}).get("total_cost", 0.0) for r in results if "cost" in r)
print(f"Total API cost: ${total_cost:.4f}")

# Save flat predictions CSV (one row per attribute) — input for RF2 export
results_df = pd.DataFrame(_build_prediction_rows(results))
results_df.to_csv(project_root / "data" / "notebook_results" / "attribute_results.csv", index=False)
print(f"Saved {len(results_df)} attribute rows to attribute_results.csv")
print(f"Columns: {results_df.columns.tolist()}")
results_df.head(10)



## Evaluation

We evaluate the pipeline results against the gold standard using a full outer join.
Each predicted attribute is matched against the expected attribute by `(concept_id_1, concept_id_2, attribute_type)`.
This gives us precision, recall, and F1 scores.

In [ ]:
from ariadne.hierarchy.evaluator import evaluate_results

eval_df = evaluate_results(results, str(gold_standard_path), cfg=cfg)

eval_df.to_csv(project_root / "data" / "notebook_results" / "attribute_evaluation.csv", index=False)

eval_df.head(20)

### Per-category breakdown

In [ ]:
n_match = (eval_df["status"] == "match").sum()
n_missed = (eval_df["status"] == "missed").sum()
n_extra = (eval_df["status"] == "extra").sum()
n_gs = n_match + n_missed
n_pred = n_match + n_extra

precision = n_match / n_pred * 100 if n_pred else 0.0
recall = n_match / n_gs * 100 if n_gs else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

print(f"Gold standard rows: {n_gs}")
print(f"Predicted rows:     {n_pred}")
print(f"Matched:            {n_match}")
print(f"Precision:          {precision:.1f}%")
print(f"Recall:             {recall:.1f}%")
print(f"F1:                 {f1:.1f}%")

breakdown = (
    eval_df.groupby("attribute_type")["status"]
    .value_counts()
    .unstack(fill_value=0)
    .reindex(columns=["match", "missed", "extra"], fill_value=0)
)
breakdown["precision%"] = (
    breakdown["match"] / (breakdown["match"] + breakdown["extra"]).clip(lower=1) * 100
).round(1)
breakdown["recall%"] = (
    breakdown["match"] / (breakdown["match"] + breakdown["missed"]).clip(lower=1) * 100
).round(1)

breakdown

### Missed and extra predictions

Inspect which attributes were missed by the pipeline and which were incorrectly predicted.

In [ ]:
missed = eval_df[eval_df["status"] == "missed"][["concept_name_1", "attribute_type", "gs_concept_name_2"]]
print(f"Missed predictions: {len(missed)}")
missed.head(20)

In [ ]:
extra = eval_df[eval_df["status"] == "extra"][["concept_name_1", "attribute_type", "predicted_concept_name_2"]]
print(f"Extra predictions: {len(extra)}")
extra.head(20)

## RF2 Delta Export

Convert the predicted attributes to SNOMED CT RF2 delta files for use with the ELK reasoner.
Source OMOP concept IDs are remapped to synthetic SCTIDs (starting at 1 000 000 001)
to avoid collisions with real SNOMED IDs in the base release.

Concept definitions are expressed as **OWL Functional Syntax axioms** in the OWL Axiom Reference Set,
which gives ELK more precise subsumption semantics than the legacy StatedRelationship format:

- A concept with attributes becomes `EquivalentClasses(:src ObjectIntersectionOf(:parent ObjectSomeValuesFrom(:609096000 ...)))`
- A concept with no attributes falls back to `SubClassOf(:src :parent)`

The output ZIP contains files under `Delta/`:
- `Delta/Terminology/sct2_Concept_Delta_INT_{date}.txt` — one row per source term (sufficiently defined)
- `Delta/Terminology/sct2_StatedRelationship_Delta_INT_{date}.txt` — header-only placeholder
- `Delta/Terminology/sct2_Relationship_Delta_INT_{date}.txt` — header-only placeholder (required by toolkit)
- `Delta/Refset/Content/der2_sRefset_OWLAxiomDelta_INT_{date}.txt` — OWL axiom per source concept
- `Delta/Refset/Metadata/der2_ssRefset_ModuleDependencyDelta_INT_{date}.txt` — module dependency

In [6]:

# ── Stated parent selection via reference-term neighbourhood voting ────────
import importlib
import json
import psycopg
import ariadne.hierarchy.parent_selector as _ps_mod
importlib.reload(_ps_mod)
from ariadne.hierarchy.parent_selector import build_stated_parents_map
from ariadne.utils.utils import get_environment_variable

conn_str = get_environment_variable("VOCAB_CONNECTION_STRING")
conn_str = conn_str.replace("+psycopg", "").replace("+psycopg2", "")
schema = get_environment_variable("VOCAB_SCHEMA")


# Load raw hierarchy results (contains reference_examples with concept_id + similarity)
raw_results_path = project_root / "data" / "notebook_results" / "hierarchy_results_raw.json"
with open(raw_results_path) as fh:
    raw_results = json.load(fh)

print(f"Loaded {len(raw_results)} raw result entries.")

# ── Tuning parameters ─────────────────────────────────────────────────
#   top_k=1: single best parent (reduces ancestor-explosion false positives)
#   min_similarity=0.7: minimum cosine similarity to count a reference term
#   use_attr_filter=True: skip reference terms less specific than the source
PARENT_TOP_K = 2
PARENT_MIN_SIM = 0.6

with psycopg.connect(conn_str) as parent_conn:
    stated_parents_map = build_stated_parents_map(
        raw_results,
        parent_conn,
        schema,
        top_k=PARENT_TOP_K,
        min_similarity=PARENT_MIN_SIM,
        use_attr_filter=True,
    )

# Coverage report
fallback = [v for v in stated_parents_map.values() if v == ["404684003"]]
specific = len(stated_parents_map) - len(fallback)
print(f"\nStated parent coverage (top_k={PARENT_TOP_K}, min_sim={PARENT_MIN_SIM}):")
print(f"  Concepts with specific parent(s) : {specific}/{len(stated_parents_map)}")
print(f"  Fell back to Clinical finding    : {len(fallback)}/{len(stated_parents_map)}")

# Sample 5 concepts with their selected parents
sample_ids = list(stated_parents_map)[:5]
for cid in sample_ids:
    src_name = next((e.get("source_concept_name","?") for e in raw_results
                     if e.get("source_concept_id") == cid), "?")
    print(f"  {src_name!r:50s} → {stated_parents_map[cid]}")


Loaded 342 raw result entries.

Stated parent coverage (top_k=2, min_sim=0.6):
  Concepts with specific parent(s) : 321/341
  Fell back to Clinical finding    : 20/341
  'Gestation period, 17 weeks'                       → ['428567001', '59466002']
  'Plasmacytoma in remission'                        → ['415112005', '105603000']
  'End-stage renal disease'                          → ['46177005', '709044004']
  'Cellulitis of leg, excluding foot'                → ['1297310008', '1297307001']
  'Chronic kidney disease stage 5'                   → ['709044004', '431855005']


In [7]:
import importlib, sys, shutil

# Force a fresh import — drop ALL ariadne.hierarchy modules from cache
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("ariadne"):
        del sys.modules[mod_name]

from ariadne.hierarchy.rf2_exporter import export_to_rf2

rf2_output_dir = project_root / "data" / "rf2_output"
# Clean previous delta files to avoid stale artifacts
old_delta = rf2_output_dir / "SnomedCT_YourDelta"
if old_delta.exists():
    shutil.rmtree(old_delta)
old_delta2 = rf2_output_dir / "Delta"
if old_delta2.exists():
    shutil.rmtree(old_delta2)

zip_path, id_map = export_to_rf2(
    source=project_root / "data" / "notebook_results" / "attribute_results.csv",
    output_dir=rf2_output_dir,
    stated_parent=stated_parents_map,   # per-concept parents from neighbourhood voting
)
print(f"RF2 delta written to: {zip_path}")
print(f"ID mapping: {len(id_map)} concepts (OMOP → synthetic)")

# Verify OWL content inline
import zipfile
with zipfile.ZipFile(zip_path) as zf:
    owl_name = next(n for n in zf.namelist() if "OWLAxiom" in n)
    with zf.open(owl_name) as f:
        owl_lines = f.read().decode("utf-8").splitlines()
print(f"OWL axiom rows: {len(owl_lines) - 1}")
if owl_lines[1:]:
    sample = owl_lines[1].split("\t")
    print(f"Sample expression: {sample[-1][:150]}")
id_map.head()


RF2 delta written to: /Users/aostrop1/Library/CloudStorage/OneDrive-JNJ/projects/aostropolets_git/Ariadne/data/rf2_output/snomed_delta_20260405.zip
ID mapping: 341 concepts (OMOP → synthetic)
OWL axiom rows: 341
Sample expression: EquivalentClasses(:1000000001 ObjectIntersectionOf(:428567001 :59466002 ObjectSomeValuesFrom(:609096000 ObjectSomeValuesFrom(:246454002 :1156670009)))


,omop_concept_id,synthetic_sctid,concept_name
0,4277749,1000000001,"Gestation period, 17 weeks"
1,1450702,1000000002,Plasmacytoma in remission
2,193782,1000000003,End-stage renal disease
3,4113790,1000000004,"Cellulitis of leg, excluding foot"
4,443611,1000000005,Chronic kidney disease stage 5


## SNOMED Classification (ELK Reasoner)

Run the ELK OWL EL++ classifier (via [snomed-owl-toolkit](https://github.com/IHTSDO/snomed-owl-toolkit))
to infer **Is a** (parent) relationships from the predicted attributes.

The classifier takes the RF2 delta ZIP produced above and a base SNOMED CT International Edition
snapshot, converts both to OWL, runs ELK, and returns the inferred parent relationships.

### Prerequisites
- **Java 17+** installed and on PATH
- **snomed-owl-toolkit JAR** — download from [GitHub releases](https://github.com/IHTSDO/snomed-owl-toolkit/releases)
- **SNOMED CT International Edition RF2 snapshot** — obtain from [MLDS](https://mlds.ihtsdotools.org/)

### Pre-classification validation

Check the RF2 delta for structural issues before sending it to the classifier.

In [8]:
from ariadne.hierarchy.classifier import (
    classify_delta,
    classification_summary,
    parse_classification_results,
    pre_classification_checks,
    resolve_parent_names,
)

delta_zip = project_root / "data" / "rf2_output" / next(
    f.name for f in (project_root / "data" / "rf2_output").iterdir()
    if f.name.startswith("snomed_delta_") and f.name.endswith(".zip")
)
print(f"Delta ZIP: {delta_zip}")

issues = pre_classification_checks(delta_zip)
if issues:
    print("⚠️  Issues found:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("✅ All pre-classification checks passed.")

Delta ZIP: /Users/aostrop1/Library/CloudStorage/OneDrive-JNJ/projects/aostropolets_git/Ariadne/data/rf2_output/snomed_delta_20260405.zip
✅ All pre-classification checks passed.


### Run classification

This step takes ~90-120 seconds (dominated by loading the base SNOMED release).
The snomed-owl-toolkit converts RF2 → OWL, runs ELK, and produces a results ZIP
with the inferred "Is a" relationships.

In [9]:
results_zip = classify_delta(
    delta_zip,
    #base_snomed_zip=str(project_root / "data" / "SnomedCT_InternationalRF2.zip"),
    base_snomed_zip=str(project_root / "data" / "SnomedCT_InternationalRF2.zip"),
    toolkit_jar=str(project_root / "tools" / "snomed-owl-toolkit-5.3.0-executable.jar"),
    output_dir=project_root / "data" / "rf2_output",
)
print(f"Classification results: {results_zip}")

Classification results: /Users/aostrop1/Library/CloudStorage/OneDrive-JNJ/projects/aostropolets_git/Ariadne/data/rf2_output/classification-results-2026-04-05_14-49-21.zip


### Parse results and resolve names

Extract the new inferred "Is a" relationships and resolve SNOMED concept names.

In [12]:

new_is_a, removed = parse_classification_results(results_zip)

print(f"New inferred 'Is a' relationships: {len(new_is_a)}")
print(f"Redundant relationships removed:   {len(removed)}")

# Build lookup structures
synth_to_omop = dict(zip(
    id_map["synthetic_sctid"].astype(str),
    id_map["omop_concept_id"].astype(str),
))
synth_ids = set(id_map["synthetic_sctid"].astype(str))

# src_name_map: OMOP ID → concept name (from id_map + gold_standard override)
src_name_map = {}
for _, m in id_map.iterrows():
    omop_str = str(m["omop_concept_id"])
    if str(m.get("concept_name", "")):
        src_name_map[omop_str] = str(m["concept_name"])
for _, gs_row in gold_standard.iterrows():
    src_name_map[str(gs_row["concept_id_1"])] = gs_row["concept_name_1"]

# Collect all real SCTIDs that appear as ELK results (parents or children)
real_sctids = set()
for _, row in new_is_a.iterrows():
    src, dest = str(row["sourceId"]), str(row["destinationId"])
    src_is_synth = src in synth_ids
    dest_is_synth = dest in synth_ids
    if src_is_synth and not dest_is_synth:   # "Is a": our concept → real parent
        real_sctids.add(dest)
    elif not src_is_synth and dest_is_synth: # "Subsumes": real child → our concept
        real_sctids.add(src)

# Crosswalk: SCTID (concept_code) → (omop_concept_id, concept_name)
# constrained to vocabulary_id = 'SNOMED'
sctid_omop_map: dict[str, str] = {}   # sctid → omop concept_id (str)
sctid_name_map: dict[str, str] = {}   # sctid → concept_name
if real_sctids:
    with psycopg.connect(conn_str) as conn:
        with conn.cursor() as cur:
            cur.execute(
                f"""
                SELECT concept_code, concept_id, concept_name
                FROM {schema}.concept
                WHERE vocabulary_id = 'SNOMED'
                  AND concept_code = ANY(%s)
                """,
                (list(real_sctids),),
            )
            for code, omop_id, name in cur.fetchall():
                sctid_omop_map[str(code)] = str(omop_id)
                sctid_name_map[str(code)] = name

unmapped = real_sctids - set(sctid_omop_map)
if unmapped:
    print(f"⚠️  {len(unmapped)} SCTIDs not found in concept table (will keep SCTID as fallback)")

# Build bidirectional parents_df
#   concept_id_1 / concept_name_1 : always the GS/pipeline concept (OMOP ID)
#   concept_id_2 / concept_name_2 : always the SNOMED concept from ELK, mapped to OMOP concept_id
#   relationship_id               : "Is a"     → our concept is a child of the SNOMED concept
#                                   "Subsumes" → our concept is a parent of the SNOMED concept
rows = []
skipped_inter_delta = 0
skipped_base_only   = 0
for _, row in new_is_a.iterrows():
    src, dest = str(row["sourceId"]), str(row["destinationId"])
    src_is_synth = src in synth_ids
    dest_is_synth = dest in synth_ids

    if src_is_synth and dest_is_synth:   # inter-delta → skip
        skipped_inter_delta += 1
        continue
    if not src_is_synth and not dest_is_synth:  # base-only → skip
        skipped_base_only += 1
        continue

    if src_is_synth and not dest_is_synth:
        # Our concept inferred under a real SNOMED parent → "Is a"
        omop_id = synth_to_omop[src]
        rows.append({
            "concept_id_1":   omop_id,
            "concept_name_1": src_name_map.get(omop_id, omop_id),
            "relationship_id": "Is a",
            "concept_id_2":   sctid_omop_map.get(dest, dest),   # OMOP concept_id via crosswalk
            "concept_name_2": sctid_name_map.get(dest, dest),
        })
    else:
        # Real SNOMED concept subsumed under our new concept → "Subsumes"
        omop_id = synth_to_omop[dest]
        rows.append({
            "concept_id_1":   omop_id,
            "concept_name_1": src_name_map.get(omop_id, omop_id),
            "relationship_id": "Subsumes",
            "concept_id_2":   sctid_omop_map.get(src, src),     # OMOP concept_id via crosswalk
            "concept_name_2": sctid_name_map.get(src, src),
        })

parents_df = pd.DataFrame(rows, columns=[
    "concept_id_1", "concept_name_1", "relationship_id",
    "concept_id_2", "concept_name_2",
])

parents_csv = project_root / "data" / "notebook_results" / "classification_parents.csv"
parents_df.to_csv(parents_csv, index=False)

print(f"\nSkipped inter-delta rows : {skipped_inter_delta}")
print(f"Skipped base-only rows   : {skipped_base_only}")
print(f"Saved {len(parents_df)} parent relationships → classification_parents.csv")
print(f"  'Is a'     (our concept → real SNOMED parent) : {(parents_df['relationship_id']=='Is a').sum()}")
print(f"  'Subsumes' (real SNOMED child → our concept)  : {(parents_df['relationship_id']=='Subsumes').sum()}")
parents_df.head(20)


EQUIVALENT CONCEPTS FOUND (96 rows)!  This usually indicates a modelling error — two concepts have identical defining attributes.


New inferred 'Is a' relationships: 1303
Redundant relationships removed:   556

Skipped inter-delta rows : 52
Skipped base-only rows   : 0
Saved 1251 parent relationships → classification_parents.csv
  'Is a'     (our concept → real SNOMED parent) : 758
  'Subsumes' (real SNOMED child → our concept)  : 493


,concept_id_1,concept_name_1,relationship_id,concept_id_2,concept_name_2
0,27835,Neoplasm of uncertain behavior of pituitary gland,Subsumes,43531118,Pituicytoma
1,24602,Benign neoplasm of esophagus,Is a,4242980,Benign neoplasm of trunk
2,24602,Benign neoplasm of esophagus,Is a,24602,Benign neoplasm of esophagus
3,24602,Benign neoplasm of esophagus,Is a,442105,Benign neoplasm of soft tissue
4,72618,Disorder of skeletal muscle,Subsumes,4094735,Fluctuating muscle tone
5,72985,Arthropathy of the ankle and/or foot associate...,Subsumes,4173757,Post-infective arthritis of joint of foot
6,72991,Traumatic arthropathy of multiple sites,Is a,80806,Climacteric arthritis of multiple sites
7,74127,Contracture of elbow joint,Is a,74127,Contracture of elbow joint
8,74127,Contracture of elbow joint,Is a,764154,Disorder of left upper extremity
9,72745,Open dislocation of elbow,Subsumes,4012604,Open traumatic subluxation of elbow


## Evaluation: inferred parents vs concept_relationship

Compare the ELK-inferred "Is a" parents against the actual direct "Is a" relationships
from the `concept_relationship` table in the vocabulary database.

- **Predicted**: new inferred "Is a" from ELK classification results
- **Actual**: direct "Is a" parents from `concept_relationship` (filtered to active, `relationship_id = 'Is a'`)

Both are matched on `(source_concept_id, parent_sctid)` pairs.

In [14]:

# ── Evaluation: Is a only ──────────────────────────────────────────────────
# Predicted: "Is a" rows from parents_df (our concept → SNOMED parent, both as OMOP IDs)
# Actual:    direct "Is a" parents from concept_relationship, both sides as OMOP concept_id

source_ids = gold_standard["concept_id_1"].unique().tolist()

# Query actual "Is a" parents — return both sides as OMOP concept_ids
with psycopg.connect(conn_str) as conn:
    with conn.cursor() as cur:
        cur.execute(
            f"""
            SELECT cr.concept_id_1,
                   cr.concept_id_2      AS parent_omop_id,
                   c2.concept_name      AS parent_name
            FROM {schema}.concept_relationship cr
            JOIN {schema}.concept c2
              ON c2.concept_id = cr.concept_id_2
            WHERE cr.concept_id_1 = ANY(%s)
              AND cr.relationship_id = 'Is a'
              AND cr.invalid_reason IS NULL
              AND c2.vocabulary_id  = 'SNOMED'
            """,
            (source_ids,),
        )
        actual_rows = cur.fetchall()

actual_set      = {(str(r[0]), str(r[1])) for r in actual_rows}
actual_name_map = {(str(r[0]), str(r[1])): r[2] for r in actual_rows}

# Predicted set: filter parents_df to "Is a" rows, use OMOP IDs on both sides
is_a_df = parents_df[parents_df["relationship_id"] == "Is a"]
predicted_set = {
    (str(row["concept_id_1"]), str(row["concept_id_2"]))
    for _, row in is_a_df.iterrows()
}

tp = predicted_set & actual_set
fp = predicted_set - actual_set
fn = actual_set - predicted_set

precision = len(tp) / (len(tp) + len(fp)) * 100 if (len(tp) + len(fp)) else 0.0
recall    = len(tp) / (len(tp) + len(fn)) * 100 if (len(tp) + len(fn)) else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

print(f"Actual 'Is a' pairs (from DB):   {len(actual_set)}")
print(f"Predicted 'Is a' pairs (ELK):    {len(predicted_set)}")
print(f"True positives (correct):        {len(tp)}")
print(f"False positives (extra):         {len(fp)}")
print(f"False negatives (missed):        {len(fn)}")
print(f"Precision: {precision:.1f}%")
print(f"Recall:    {recall:.1f}%")
print(f"F1:        {f1:.1f}%")


Actual 'Is a' pairs (from DB):   817
Predicted 'Is a' pairs (ELK):    758
True positives (correct):        83
False positives (extra):         675
False negatives (missed):        734
Precision: 10.9%
Recall:    10.2%
F1:        10.5%


In [15]:

# Build full parent-level evaluation: one row per (source, parent) pair
# Flag: "both" = true positive, "predicted_only" = false positive, "gs_only" = false negative
src_name_map = dict(zip(
    gold_standard["concept_id_1"].astype(str),
    gold_standard["concept_name_1"],
))
for _, m in id_map.iterrows():
    omop_str = str(m["omop_concept_id"])
    if omop_str not in src_name_map and str(m.get("concept_name", "")):
        src_name_map[omop_str] = str(m["concept_name"])

# Resolve parent OMOP IDs → names (actual + predicted), both sets already use OMOP concept_ids
all_parent_omop_ids = list({p for _, p in actual_set | predicted_set})
parent_name_lookup: dict[str, str] = {str(r[0]): r[2] for r in actual_rows}  # seed from actual
with psycopg.connect(conn_str) as conn:
    with conn.cursor() as cur:
        cur.execute(
            f"SELECT concept_id, concept_name FROM {schema}.concept "
            f"WHERE concept_id = ANY(%s)",
            ([int(x) for x in all_parent_omop_ids],),
        )
        for omop_id, name in cur.fetchall():
            parent_name_lookup[str(omop_id)] = name

eval_rows = []
for sid, pid in tp:
    eval_rows.append({
        "concept_id_1":  sid,
        "concept_name":  src_name_map.get(sid, sid),
        "parent_omop_id": pid,
        "parent_name":   parent_name_lookup.get(pid, pid),
        "status": "both",
    })
for sid, pid in fp:
    eval_rows.append({
        "concept_id_1":  sid,
        "concept_name":  src_name_map.get(sid, sid),
        "parent_omop_id": pid,
        "parent_name":   parent_name_lookup.get(pid, pid),
        "status": "predicted_only",
    })
for sid, pid in fn:
    eval_rows.append({
        "concept_id_1":  sid,
        "concept_name":  src_name_map.get(sid, sid),
        "parent_omop_id": pid,
        "parent_name":   parent_name_lookup.get(pid, pid),
        "status": "gs_only",
    })

is_a_eval_df = pd.DataFrame(eval_rows).sort_values(["concept_name", "status", "parent_name"])
is_a_eval_df.to_csv(
    project_root / "data" / "notebook_results" / "classification_is_a_evaluation.csv",
    index=False,
)

n_both      = (is_a_eval_df["status"] == "both").sum()
n_pred_only = (is_a_eval_df["status"] == "predicted_only").sum()
n_gs_only   = (is_a_eval_df["status"] == "gs_only").sum()
print(f"Total rows: {len(is_a_eval_df)}")
print(f"  both (TP):           {n_both}")
print(f"  predicted_only (FP): {n_pred_only}")
print(f"  gs_only (FN):        {n_gs_only}")
print(f"\nSaved → classification_is_a_evaluation.csv")
is_a_eval_df.head(30)


Total rows: 1492
  both (TP):           83
  predicted_only (FP): 675
  gs_only (FN):        734

Saved → classification_is_a_evaluation.csv


,concept_id_1,concept_name,parent_omop_id,parent_name,status
61,73026,Abnormal breath sounds,4278456,Breath sounds - finding,both
716,73026,Abnormal breath sounds,4027547,Respiratory auscultation finding,predicted_only
1236,73361,Abrasion and/or friction burn of elbow with in...,43530817,"Abrasion and/or friction burn of upper limb, i...",gs_only
922,73361,Abrasion and/or friction burn of elbow with in...,4106352,Superficial injury of elbow,gs_only
1112,73361,Abrasion and/or friction burn of elbow with in...,444193,Superficial injury of elbow with infection,gs_only
276,73361,Abrasion and/or friction burn of elbow with in...,73361,Abrasion and/or friction burn of elbow with in...,predicted_only
1258,31609,Abscess of salivary gland,444271,Abscess of oral tissue,gs_only
1287,31609,Abscess of salivary gland,4114483,Lesion of salivary gland,gs_only
1433,31609,Abscess of salivary gland,4322566,Mass of salivary gland,gs_only
478,31609,Abscess of salivary gland,4061734,Abscess of face,predicted_only
